# TwinChain — Modelo de línea base + banda (Olist)
Sigue `GUIA_ENTRENAMIENTO_LINEA_BASE`. Regresión cuantílica (GradientBoostingRegressor) + Conformalized Quantile Regression (CQR).

**RF017:** las features salen solo de la Fase I propia del vendedor. Por eso este notebook usa únicamente
`olist_orders_dataset.csv` y `olist_order_items_dataset.csv`; los demás CSV de Olist (clientes, geolocalización,
productos/categoría, pagos, reviews, vendedores) son contexto externo y se dejan afuera a propósito.

## Paso 1 — Datos

In [1]:
import os
print(sorted(os.listdir('data/')))

['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


## Paso 2 — Cargar y explorar (EDA)

In [2]:
import pandas as pd
import numpy as np

orders = pd.read_csv(
    'data/olist_orders_dataset.csv',
    parse_dates=[
        'order_purchase_timestamp', 'order_approved_at',
        'order_delivered_carrier_date', 'order_delivered_customer_date',
        'order_estimated_delivery_date',
    ],
)
items = pd.read_csv(
    'data/olist_order_items_dataset.csv',
    parse_dates=['shipping_limit_date'],
)
print(f"orders: {orders.shape[0]:,} filas, {orders.shape[1]} columnas")
print(f"items:  {items.shape[0]:,} filas, {items.shape[1]} columnas")
orders.head(3)

orders: 99,441 filas, 8 columnas
items:  112,650 filas, 7 columnas


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04


In [3]:
print(orders['order_status'].value_counts())
print(f"\nCon order_delivered_carrier_date no nula: "
      f"{orders['order_delivered_carrier_date'].notna().sum():,}")

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Con order_delivered_carrier_date no nula: 97,658


In [4]:
items.head(3)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


## Paso 3 — Target: `margin_days`
`margin_days = shipping_limit_date - order_delivered_carrier_date` (días). Positivo = despachó antes de su compromiso; negativo = tarde.

In [5]:
delivered = orders[
    (orders['order_status'] == 'delivered')
    & orders['order_delivered_carrier_date'].notna()
]
merged = items.merge(
    delivered[['order_id', 'order_delivered_carrier_date']],
    on='order_id', how='inner',
)
merged['margin_days'] = (
    merged['shipping_limit_date'] - merged['order_delivered_carrier_date']
).dt.total_seconds() / 86400.0
print(merged['margin_days'].describe())

count    110195.000000
mean          3.319604
std           5.827487
min        -116.762569
25%           1.532292
50%           3.215035
75%           5.204965
max        1046.855324
Name: margin_days, dtype: float64


## Paso 4 — Descartar anomalías de calidad de dato (|margin_days| > 60)

In [6]:
OUTLIER_ABS_DAYS = 60
n_before = len(merged)
merged = merged[merged['margin_days'].abs() <= OUTLIER_ABS_DAYS].copy()
n_dropped = n_before - len(merged)
print(f"Filas descartadas por |margin_days| > {OUTLIER_ABS_DAYS} días: {n_dropped}")

Filas descartadas por |margin_days| > 60 días: 8


## Paso 5 — Agregación semanal por vendedor (RF010/RF011)
Semana calendario (lunes) de la fecha real de despacho. `.dt.start_time` es equivalente al `.apply(lambda p: p.start_time)` de la guía, pero vectorizado.

In [7]:
merged['week'] = merged['order_delivered_carrier_date'].dt.to_period('W').dt.start_time
weekly = (
    merged.groupby(['seller_id', 'week'])['margin_days']
    .agg(['mean', 'count'])
    .reset_index()
)
weekly.columns = ['seller_id', 'week', 'margin_mean', 'n_items']
weekly = weekly.sort_values(['seller_id', 'week']).reset_index(drop=True)
print(f"Vendedores distintos con al menos una semana de actividad: "
      f"{weekly['seller_id'].nunique():,}")
weekly.head(10)

Vendedores distintos con al menos una semana de actividad: 2,970


,seller_id,week,margin_mean,n_items
0,0015a82c2db000af6aaaf3ae2ecb0532,2017-09-25,13.271678,1
1,0015a82c2db000af6aaaf3ae2ecb0532,2017-10-16,2.678466,2
2,001cca7ae9ae17fb1caed9dfb1094831,2017-02-06,-1.596192,1
3,001cca7ae9ae17fb1caed9dfb1094831,2017-02-20,1.903681,6
4,001cca7ae9ae17fb1caed9dfb1094831,2017-02-27,5.996080,6
5,001cca7ae9ae17fb1caed9dfb1094831,2017-03-06,2.248507,1
6,001cca7ae9ae17fb1caed9dfb1094831,2017-03-20,1.768931,5
7,001cca7ae9ae17fb1caed9dfb1094831,2017-03-27,2.007723,8
8,001cca7ae9ae17fb1caed9dfb1094831,2017-04-03,2.444194,3
9,001cca7ae9ae17fb1caed9dfb1094831,2017-04-10,4.740361,6


## Paso 6 — Fase I / Fase II (Montgomery, sec. 5.3.7)
Fase I = primeras 4 semanas activas (línea base fija). Fase II = el resto (target: su media).

In [8]:
PHASE1_N_WEEKS = 4
MIN_PHASE1_WEEKS = 4
MIN_PHASE2_WEEKS = 2

rows = []
discarded_short_phase1 = 0
discarded_no_phase2 = 0
for seller_id, g in weekly.groupby('seller_id'):
    g = g.sort_values('week').reset_index(drop=True)
    if len(g) < MIN_PHASE1_WEEKS:
        discarded_short_phase1 += 1
        continue
    phase1 = g.iloc[:PHASE1_N_WEEKS]
    phase2 = g.iloc[PHASE1_N_WEEKS:]
    if len(phase1) < MIN_PHASE1_WEEKS or len(phase2) < MIN_PHASE2_WEEKS:
        discarded_no_phase2 += 1
        continue
    rows.append({
        'seller_id': seller_id,
        'phase1_mean': phase1['margin_mean'].mean(),
        'phase1_std': phase1['margin_mean'].std(ddof=0),
        'phase1_min': phase1['margin_mean'].min(),
        'phase1_max': phase1['margin_mean'].max(),
        'phase1_mean_items_per_week': phase1['n_items'].mean(),
        'phase1_total_items': phase1['n_items'].sum(),
        'phase1_n_weeks': len(phase1),
        'phase2_mean': phase2['margin_mean'].mean(),
        'phase2_n_weeks': len(phase2),
    })

feat = pd.DataFrame(rows)
feat['phase1_std'] = feat['phase1_std'].fillna(0.0)
print(f"Vendedores usables: {len(feat):,}")
print(f"Descartados por Fase I corta (< 4 semanas): {discarded_short_phase1:,}")
print(f"Descartados por no llegar a Fase II (< 2 semanas): {discarded_no_phase2:,}")

Vendedores usables: 1,460
Descartados por Fase I corta (< 4 semanas): 1,172
Descartados por no llegar a Fase II (< 2 semanas): 338


## Paso 7 — Features (RF017: solo Fase I propia)

In [9]:
FEATURE_COLS = [
    'phase1_mean', 'phase1_std', 'phase1_min', 'phase1_max',
    'phase1_mean_items_per_week', 'phase1_total_items', 'phase1_n_weeks',
]
feat[FEATURE_COLS + ['phase2_mean']].describe().round(2)

,phase1_mean,phase1_std,phase1_min,phase1_max,phase1_mean_items_per_week,phase1_total_items,phase1_n_weeks,phase2_mean
count,1460.00,1460.00,1460.00,1460.00,1460.00,1460.00,1460.0,1460.00
mean,3.36,1.97,0.61,5.60,1.98,7.92,4.0,3.18
std,2.80,2.15,5.48,2.70,1.61,6.42,0.0,2.34
min,-18.24,0.05,-58.59,-3.95,1.00,4.00,4.0,-27.28
25%,2.41,0.96,0.03,4.47,1.25,5.00,4.0,2.32
50%,3.63,1.41,1.87,5.36,1.50,6.00,4.0,3.24
75%,4.58,2.07,2.95,6.11,2.25,9.00,4.0,3.97
max,17.96,25.38,14.09,32.92,26.50,106.00,4.0,18.03


## Paso 8 — Split train / calibración / test (60/20/20)

In [10]:
RNG_SEED = 42
TRAIN_FRAC, CALIB_FRAC = 0.6, 0.2

rng = np.random.RandomState(RNG_SEED)
idx = feat.index.to_numpy().copy()
rng.shuffle(idx)
n = len(idx)
n_train = int(n * TRAIN_FRAC)
n_calib = int(n * CALIB_FRAC)

train_df = feat.loc[idx[:n_train]].reset_index(drop=True)
calib_df = feat.loc[idx[n_train:n_train + n_calib]].reset_index(drop=True)
test_df  = feat.loc[idx[n_train + n_calib:]].reset_index(drop=True)
print(f"Train: {len(train_df)} | Calibración: {len(calib_df)} | Test: {len(test_df)}")

Train: 876 | Calibración: 292 | Test: 292


## Paso 9 — Entrenar 3 modelos de cuantil (0.1 / 0.5 / 0.9)

In [11]:
from sklearn.ensemble import GradientBoostingRegressor

QUANTILES = (0.1, 0.5, 0.9)
X_train = train_df[FEATURE_COLS].to_numpy()
y_train = train_df['phase2_mean'].to_numpy()

models = {}
for q in QUANTILES:
    gbr = GradientBoostingRegressor(
        loss='quantile', alpha=q,
        n_estimators=150, max_depth=3, learning_rate=0.05,
        subsample=0.8, random_state=RNG_SEED,
    )
    gbr.fit(X_train, y_train)
    models[q] = gbr
    print(f"Modelo cuantil {q} entrenado")

Modelo cuantil 0.1 entrenado
Modelo cuantil 0.5 entrenado


Modelo cuantil 0.9 entrenado


## Paso 10 — Calibrar la banda con CQR (Romano, Patterson y Candès, 2019)

In [12]:
X_calib = calib_df[FEATURE_COLS].to_numpy()
y_calib = calib_df['phase2_mean'].to_numpy()

q_lo = models[0.1].predict(X_calib)
q_hi = models[0.9].predict(X_calib)
scores = np.maximum(q_lo - y_calib, y_calib - q_hi)

nominal_coverage = 0.9 - 0.1
n = len(scores)
level = min(np.ceil((n + 1) * nominal_coverage) / n, 1.0)
Q = np.quantile(scores, level, method='higher')
print(f"Ajuste conforme Q = {Q:.3f} días")

Ajuste conforme Q = 0.299 días


## Paso 11 — Evaluar en test

In [13]:
from sklearn.metrics import mean_absolute_error
from scipy import stats

X_test = test_df[FEATURE_COLS].to_numpy()
y_test = test_df['phase2_mean'].to_numpy()

pred_median = models[0.5].predict(X_test)
pred_lo_raw = models[0.1].predict(X_test)
pred_hi_raw = models[0.9].predict(X_test)
pred_lo_cqr = pred_lo_raw - Q
pred_hi_cqr = pred_hi_raw + Q

mae_modelo = mean_absolute_error(y_test, pred_median)
mae_ingenuo = mean_absolute_error(y_test, test_df['phase1_mean'].to_numpy())
print(f"MAE del modelo:        {mae_modelo:.2f} días")
print(f"MAE línea base ingenua (media cruda de Fase I, sin ML): {mae_ingenuo:.2f} días")

coverage_raw = np.mean((y_test >= pred_lo_raw) & (y_test <= pred_hi_raw))
coverage_cqr = np.mean((y_test >= pred_lo_cqr) & (y_test <= pred_hi_cqr))
print(f"\nCobertura banda 80% — sin CQR: {coverage_raw:.1%}")
print(f"Cobertura banda 80% — con CQR: {coverage_cqr:.1%}")

n_test = len(y_test)
p_raw = stats.binomtest(int(round(coverage_raw * n_test)), n_test, 0.8).pvalue
p_cqr = stats.binomtest(int(round(coverage_cqr * n_test)), n_test, 0.8).pvalue
print(f"\np-value vs. 80% nominal — sin CQR: {p_raw:.4f} "
      f"({'sí' if p_raw < 0.05 else 'no'} es significativamente distinta)")
print(f"p-value vs. 80% nominal — con CQR: {p_cqr:.4f} "
      f"({'sí' if p_cqr < 0.05 else 'no'} es significativamente distinta)")
print(f"\nAncho medio de banda CQR: {np.mean(pred_hi_cqr - pred_lo_cqr):.2f} días")

MAE del modelo:        1.36 días
MAE línea base ingenua (media cruda de Fase I, sin ML): 1.83 días

Cobertura banda 80% — sin CQR: 69.9%
Cobertura banda 80% — con CQR: 79.1%

p-value vs. 80% nominal — sin CQR: 0.0000 (sí es significativamente distinta)
p-value vs. 80% nominal — con CQR: 0.7145 (no es significativamente distinta)

Ancho medio de banda CQR: 3.81 días


## Paso 12 — Guardar el modelo

In [14]:
import joblib

bundle = {
    'model_family': 'GradientBoostingRegressor (scikit-learn)',
    'uncertainty_technique': 'Conformalized Quantile Regression (CQR)',
    'quantile_models': models,
    'cqr_adjustment_Q_days': Q,
    'quantiles': {'lo': 0.1, 'median': 0.5, 'hi': 0.9},
    'feature_columns': FEATURE_COLS,
    'phase1_n_weeks': PHASE1_N_WEEKS,
    'min_phase1_weeks': MIN_PHASE1_WEEKS,
    'min_phase2_weeks': MIN_PHASE2_WEEKS,
    'outlier_abs_days_threshold': OUTLIER_ABS_DAYS,
    'training_dataset': 'Olist (olist_orders_dataset.csv + olist_order_items_dataset.csv)',
    'target_definition': (
        'margin_days = shipping_limit_date - order_delivered_carrier_date (dias); '
        'positivo=despacho antes del compromiso propio del vendedor, '
        'negativo=despacho tardio.'
    ),
    'sklearn_hyperparameters': {
        'n_estimators': 150, 'max_depth': 3, 'learning_rate': 0.05,
        'subsample': 0.8, 'random_state': RNG_SEED,
    },
}
joblib.dump(bundle, 'baseline_model_bundle.joblib')
print("Modelo guardado en baseline_model_bundle.joblib")

Modelo guardado en baseline_model_bundle.joblib


In [15]:
loaded = joblib.load('baseline_model_bundle.joblib')
sample = test_df[FEATURE_COLS].iloc[[0]].to_numpy()
pred = loaded['quantile_models'][0.5].predict(sample)[0]
print(f"Predicción de prueba tras recargar el modelo: {pred:.2f} días")
print(test_df[FEATURE_COLS].iloc[0].to_dict())

Predicción de prueba tras recargar el modelo: 3.23 días
{'phase1_mean': 4.056028645833333, 'phase1_std': 1.787211846678734, 'phase1_min': 1.5864351851851852, 'phase1_max': 6.103790509259259, 'phase1_mean_items_per_week': 1.75, 'phase1_total_items': 7.0, 'phase1_n_weeks': 4.0}
